# Aggregate node-indicator information gain

Average each node indicator's information gain across China, Cuba, Iran, Russia, UAE, and Venezuela, then export a descending LaTeX table with three decimal places.

In [2]:
from pathlib import Path

import pandas as pd

ModuleNotFoundError: No module named 'pandas'

In [ ]:
def find_project_root(start_path):
    """Find the Backbone-Graph project root containing AGENTS.md.

    Args:
        start_path: Directory from which to begin the search.

    Returns:
        Path to the Backbone-Graph project root.
    """
    start_path = Path(start_path).resolve()
    for directory in (start_path, *start_path.parents):
        if (directory / "AGENTS.md").exists():
            return directory
    raise FileNotFoundError("Could not find the Backbone-Graph project root.")

In [ ]:
campaigns = ["China_1", "Cuba", "Iran_1", "Russia_1", "UAE", "Venezuela"]
project_root = find_project_root(Path.cwd())
results_root = project_root / "results" / "labeled_datasets"

campaign_frames = []
for campaign in campaigns:
    csv_path = (
        results_root
        / campaign
        / f"{campaign}_part_all"
        / "step_3_key_authors_graph"
        / "topic_col_BERTopic_topic_256_top_percentage_100"
        / "node_indicators"
        / "node_indicators_information_gain.csv"
    )
    campaign_df = (
        pd.read_csv(csv_path, index_col=0)
        .rename_axis("indicator")
        .reset_index()
    )
    campaign_df["campaign"] = campaign
    campaign_frames.append(campaign_df)

all_campaigns_df = pd.concat(campaign_frames, ignore_index=True)
all_campaigns_df

In [ ]:
aggregated_df = (
    all_campaigns_df.groupby("indicator", as_index=False)["information_gain"]
    .mean()
    .rename(columns={"information_gain": "mean_information_gain"})
    .sort_values("mean_information_gain", ascending=False)
    .reset_index(drop=True)
)

aggregated_df.round(3)

In [ ]:
output_dir = (
    project_root.parent
    / "paper_overleaf"
    / "ComplexNetworksTemplate"
    / "figs"
)
output_dir.mkdir(parents=True, exist_ok=True)
latex_path = output_dir / "aggregated_node_indicators_information_gain.tex"

aggregated_df.to_latex(
    latex_path,
    index=False,
    float_format="%.3f",
    column_format="lr",
    escape=True,
)

print(f"Saved LaTeX table: {latex_path}")